In [1]:
import pandas as pd
import numpy as np
import re
import json
import ast

In [2]:
# ================= STEP 1: Yuklash va tanishish =================
df = pd.read_csv('super_dirty_students.csv', encoding='utf-8')
print("Step 1: Yuklandi")

Step 1: Yuklandi


In [3]:
# ================= STEP 2: String tozalash =================
string_cols = df.select_dtypes(include=['object']).columns
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace(['', 'nan', 'none', 'null', 'NaN', 'None', 'NULL'], np.nan)

print("Step 2: String tozalandi")

Step 2: String tozalandi


In [4]:
# ================= STEP 3: Raqam va sanalarni tozalash =================
numeric_cols = ['age', 'score', 'gpa', 'attendance', 'money_spent']
for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(r'[^\d\.\-]', '', regex=True)
        df[col] = df[col].str.replace(',', '')
        df[col] = df[col].str.strip()
        df[col] = pd.to_numeric(df[col], errors='coerce')
        if col.lower() in ['age', 'attendance']:
            df[col] = df[col].round().astype('Int64')
        else:
            df[col] = df[col].astype(float)

# Sana ustunlari
def convert_mixed_datetime(series):
    def parse_val(x):
        if pd.isna(x):
            return pd.NaT
        s = str(x).strip()
        if s.isdigit():
            return pd.to_datetime(int(s), unit='s')
        else:
            return pd.to_datetime(s, errors='coerce')
    return series.apply(parse_val)

for col in ['date_of_join', 'event_time']:
    if col in df.columns:
        df[col] = convert_mixed_datetime(df[col])

print("Step 3: Raqam va sanalar tozalandi")

C:\Users\User\AppData\Local\Temp\ipykernel_7612\1641934095.py:23: UserWarning: Parsing dates in %d/%m/%Y %I:%M %p format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(s, errors='coerce')


Step 3: Raqam va sanalar tozalandi


In [5]:
# ================= STEP 4: Email va phone validation =================
# Email
if 'email' in df.columns:
    df['email'] = df['email'].astype(str).str.lower().str.strip()
    df['email'] = df['email'].replace(['nan', 'none', 'null', ''], np.nan)
    email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    df['email_valid'] = df['email'].apply(lambda x: bool(re.match(email_pattern, str(x))) if pd.notna(x) else False)
    df['email_status'] = df.apply(lambda row: 'valid' if row['email_valid'] else ('missing' if pd.isna(row['email']) else 'invalid_format'), axis=1)

In [6]:
# Telefon
if 'phone' in df.columns:
    df['phone_digits'] = df['phone'].astype(str).str.replace(r'\D', '', regex=True)
    df['phone_digits'] = df['phone_digits'].replace(['', 'nan', 'none'], np.nan)
    def format_uzbek_phone(digits):
        if pd.isna(digits):
            return None
        digits = str(digits)
        if len(digits) == 12 and digits.startswith('998'):
            digits = digits[3:]
        elif len(digits) == 9:
            pass
        elif len(digits) > 9:
            digits = digits[-9:]
        else:
            return None
        if len(digits) == 9:
            return f"+998 {digits[:2]} {digits[2:5]} {digits[5:7]} {digits[7:9]}"
        else:
            return None
    df['phone_formatted'] = df['phone_digits'].apply(format_uzbek_phone)
    operator_codes = {'90':'Beeline','91':'Beeline','92':'Uzbektelecom','99':'Uzbektelecom','93':'Ucell','94':'Ucell','95':'Uzmobile','97':'Uzmobile','98':'Perfectum'}
    def get_operator(phone):
        if pd.notna(phone) and len(phone) >= 13:
            code = phone[5:7]
            return operator_codes.get(code, 'Unknown')
        return None
    df['phone_operator'] = df['phone_formatted'].apply(get_operator)

print("Step 4: Email va phone validation tugadi")

Step 4: Email va phone validation tugadi


In [7]:
# ================= STEP 5: JSON parsing =================
def parse_profile_json(df, json_col='profile_json'):
    # JSON parse
    def safe_parse(x):
        if pd.isna(x) or x == '':
            return {}
        try:
            return json.loads(x)
        except:
            try:
                return ast.literal_eval(x)
            except:
                return {}
    df['_parsed'] = df[json_col].apply(safe_parse)
    # Hobbies
    df['hobbies'] = df['_parsed'].apply(lambda x: ', '.join(x.get('hobbies', [])))
    # Skills
    def get_skills(sk):
        if not sk:
            return {}
        tech = sk.get('tech', {})
        soft = sk.get('soft', [])
        return {
            'skills_tech_python': tech.get('python', 0),
            'skills_tech_excel': tech.get('excel', 0),
            'skills_tech_sql': tech.get('sql', 0),
            'skills_soft_count': len(soft) if isinstance(soft, list) else 0,
            'skills_soft_list': ', '.join(soft) if isinstance(soft, list) else ''
        }
    skills_df = df['_parsed'].apply(lambda x: get_skills(x.get('skills', {}))).apply(pd.Series)
    # Family
    def get_family(fam):
        if not fam:
            return {}
        inc = fam.get('income', {})
        return {
            'family_siblings': fam.get('siblings', 0),
            'family_income_father': inc.get('father', 0),
            'family_income_mother': inc.get('mother', 0),
            'family_income_total': inc.get('father', 0) + inc.get('mother', 0)
        }
    family_df = df['_parsed'].apply(lambda x: get_family(x.get('family', {}))).apply(pd.Series)
    # Devices
    def get_devices(devs):
        if not isinstance(devs, list):
            return {}
        result = {
            'devices_count': len(devs),
            'has_laptop': False,
            'has_phone': False,
            'latest_device_year': 0,
            'devices_types': [],
            'devices_brands': []
        }
        years = []
        for d in devs:
            if isinstance(d, dict):
                typ = d.get('type', '').lower()
                if 'laptop' in typ:
                    result['has_laptop'] = True
                if 'phone' in typ:
                    result['has_phone'] = True
                result['devices_types'].append(typ)
                result['devices_brands'].append(d.get('brand', ''))
                yr = d.get('year', 0)
                if yr:
                    years.append(yr)
        if years:
            result['latest_device_year'] = max(years)
        result['devices_types_str'] = ', '.join(result['devices_types'])
        result['devices_brands_str'] = ', '.join(result['devices_brands'])
        # list bo'lmaganlarni qaytarish
        return {k: v for k, v in result.items() if not isinstance(v, list)}
    devices_df = df['_parsed'].apply(lambda x: get_devices(x.get('devices', []))).apply(pd.Series)
    # Birlashtirish
    df = pd.concat([df, skills_df, family_df, devices_df], axis=1)
    df.drop(columns=['_parsed'], inplace=True)
    return df

df = parse_profile_json(df, 'profile_json')
print("Step 5: JSON parsing tugadi")

Step 5: JSON parsing tugadi


In [8]:
# ================= STEP 6: Address parsing =================
def parse_address(df, address_col='address_raw'):
    if address_col not in df.columns:
        print(f"'{address_col}' ustuni topilmadi. Mavjud ustunlar: {df.columns.tolist()}")
        return df
    # Asl ustunni nusxalash
    df['addr_raw'] = df[address_col].astype(str)
    # Pochta indeksi
    def find_postal(text):
        match = re.search(r'\b\d{5,6}\b', text)
        return match.group(0) if match else None
    df['addr_postal'] = df['addr_raw'].apply(find_postal)
    # Shahar ro'yxati
    cities = ['toshkent','tashkent','samarqand','samarkand','buxoro','bukhara','xorazm','urganch','andijon','andijan','fargʻona','fergana','namangan','qo‘qon','kokand','navoiy','navoi','jizzax','jizzakh','qarshi','karshi','termiz','guliston','nukus']
    cities_lower = [c.lower() for c in cities]
    def find_city(text):
        text_lower = text.lower()
        for i, city in enumerate(cities_lower):
            if city in text_lower:
                return cities[i].title()
        return None
    df['addr_city'] = df['addr_raw'].apply(find_city)
    # Tuman ro'yxati
    districts = ['yunusobod','mirzo ulugʻbek','yakkasaroy','chilonzor','olmazor','bektemir','shayxontohur','mirobod','sergeli','yangihayot','uchtepa','xadra','hamza','so‘galli','qibray','zangiota','quyichirchiq','oʻrta chirchiq','parkent','boʻstonliq']
    districts_lower = [d.lower() for d in districts]
    def find_district(text):
        text_lower = text.lower()
        match = re.search(r'([a-zа-яё]+)\s+(tuman|district)', text_lower)
        if match:
            return match.group(1).title()
        for i, dist in enumerate(districts_lower):
            if dist in text_lower:
                return districts[i].title()
        return None
    df['addr_district'] = df['addr_raw'].apply(find_district)
    # Topilmaganlarni Unknown bilan to'ldirish
    df['addr_city'] = df['addr_city'].fillna('Unknown')
    df['addr_district'] = df['addr_district'].fillna('Unknown')
    df['addr_postal'] = df['addr_postal'].fillna('Unknown')
    print("Step 6: Address parsing tugadi")
    return df

df = parse_address(df, 'address_raw')

Step 6: Address parsing tugadi


In [9]:
# ================= Saqlash =================
df.to_csv('super_dirty_students_cleaned_final.csv', index=False)
print("Barcha qadamlar tugadi. Fayl saqlandi.")

Barcha qadamlar tugadi. Fayl saqlandi.


In [10]:
# 1.1 To'liq dublikat qatorlar
duplicate_rows = df.duplicated()
print(f"To'liq dublikat qatorlar soni: {duplicate_rows.sum()}")

To'liq dublikat qatorlar soni: 0


In [11]:
# 1.2 Muhim ustunlar bo'yicha qisman dublikatlar (masalan, email, student_id)
if 'email' in df.columns:
    email_duplicates = df.duplicated(subset=['email'], keep=False).sum()
    print(f"Email bo'yicha dublikatlar soni: {email_duplicates}")

Email bo'yicha dublikatlar soni: 443


In [12]:
# Agar email dublikatlari bo'lsa, ulardan faqat birinchisini saqlab qolish
df = df.drop_duplicates(subset=['email'], keep='first')
print(f"Email dublikatlari tozalandi.")

Email dublikatlari tozalandi.


In [13]:
if 'student_id' in df.columns:
    id_duplicates = df.duplicated(subset=['student_id'], keep=False).sum()
    print(f"student_id bo'yicha dublikatlar soni: {id_duplicates}")
    df = df.drop_duplicates(subset=['student_id'], keep='first')

student_id bo'yicha dublikatlar soni: 0


In [14]:
#2. Missing values (bo‘sh qiymatlar) tahlili

# Barcha ustunlardagi bo'sh qiymatlar soni
missing_total = df.isnull().sum()
missing_percent = (missing_total / len(df)) * 100

missing_df = pd.DataFrame({
    'Bo‘sh qiymatlar soni': missing_total,
    'Foizi (%)': missing_percent.round(2)
})



In [15]:
# Faqat bo'sh qiymatlari bor ustunlarni ko'rsatish
missing_with_data = missing_df[missing_df['Bo‘sh qiymatlar soni'] > 0]
print("\n=== Bo‘sh qiymatlar tahlili ===")
print(missing_with_data.sort_values('Foizi (%)', ascending=False))


=== Bo‘sh qiymatlar tahlili ===
                      Bo‘sh qiymatlar soni  Foizi (%)
score                                  276      48.59
family_siblings                        217      38.20
family_income_father                   217      38.20
family_income_mother                   217      38.20
skills_tech_python                     217      38.20
family_income_total                    217      38.20
skills_soft_count                      217      38.20
skills_tech_sql                        217      38.20
skills_tech_excel                      217      38.20
skills_soft_list                       217      38.20
phone_formatted                        214      37.68
phone_operator                         214      37.68
phone                                  214      37.68
phone_digits                           214      37.68
name                                   190      33.45
attendance                             181      31.87
gpa                                    166      2

In [17]:
#3. Muhim ustunlardagi bo‘sh qiymatlarni to‘ldirish (fillna)

# 3.1 Raqamli ustunlar
numeric_cols = ['age', 'gpa', 'score', 'attendance']
for col in numeric_cols:
    if col in df.columns and df[col].isnull().sum() > 0:
        if col in ['age', 'attendance']:  # butun sonlar
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            print(f"{col}: {df[col].isnull().sum()} ta bo'sh qiymat mediana ({median_val}) bilan to'ldirildi.")
        else:  # float (gpa, score)
            mean_val = df[col].mean()
            df[col] = df[col].fillna(mean_val)
            print(f"{col}: {df[col].isnull().sum()} ta bo'sh qiymat o'rtacha ({mean_val:.2f}) bilan to'ldirildi.")



age: 0 ta bo'sh qiymat mediana (20.0) bilan to'ldirildi.
gpa: 0 ta bo'sh qiymat o'rtacha (12.00) bilan to'ldirildi.
score: 0 ta bo'sh qiymat o'rtacha (75.49) bilan to'ldirildi.
attendance: 0 ta bo'sh qiymat mediana (110.0) bilan to'ldirildi.


In [18]:
# 3.2 Kategoriyali ustunlar
categorical_cols = ['gender', 'course', 'status']
for col in categorical_cols:
    if col in df.columns and df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0] if not df[col].mode().empty else 'Unknown'
        df[col] = df[col].fillna(mode_val)
        print(f"{col}: {df[col].isnull().sum()} ta bo'sh qiymat '{mode_val}' bilan to'ldirildi.")



gender: 0 ta bo'sh qiymat 'fmale' bilan to'ldirildi.


In [19]:
# 3.3 Email va telefon (agar mavjud bo'lsa)
for col in ['email', 'phone']:
    if col in df.columns and df[col].isnull().sum() > 0:
        df[col] = df[col].fillna('missing')
        print(f"{col}: {df[col].isnull().sum()} ta bo'sh qiymat 'missing' bilan to'ldirildi.")

email: 0 ta bo'sh qiymat 'missing' bilan to'ldirildi.
phone: 0 ta bo'sh qiymat 'missing' bilan to'ldirildi.


In [20]:
print("\n=== Yakuniy bo'sh qiymatlar tahlili ===")
print(df.isnull().sum()[df.isnull().sum() > 0])




=== Yakuniy bo'sh qiymatlar tahlili ===
name                    190
remarks                 106
phone_digits            214
phone_formatted         214
phone_operator          214
skills_tech_python      217
skills_tech_excel       217
skills_tech_sql         217
skills_soft_count       217
skills_soft_list        217
family_siblings         217
family_income_father    217
family_income_mother    217
family_income_total     217
dtype: int64


In [32]:
#Step 8: Data normalization
#1. Gender normallashtirish

def normalize_gender(df, col='gender'):
    if col not in df.columns:
        print(f"'{col}' ustuni topilmadi.")
        return df
    
    # Mavjud qiymatlarni ko'rish
    print(f"Gender unikal qiymatlari (oldin): {df[col].unique()}")
    
    # Mapping lug'ati
    gender_map = {
        'm': 'Male', 'male': 'Male', 'man': 'Male', 'мужской': 'Male', 'erkak': 'Male',
        'f': 'Female', 'female': 'Female', 'woman': 'Female', 'женский': 'Female', 'ayol': 'Female',
        'other': 'Unknown', 'unknown': 'Unknown', '': 'Unknown', 'nan': 'Unknown', 'none': 'Unknown'
    }
    
    # Kichik harf va probellarni tozalash
    df[col] = df[col].astype(str).str.lower().str.strip()
    
    # Map orqali o'zgartirish, topilmaganlarni 'Unknown' qilish
    df[col] = df[col].map(gender_map).fillna('Unknown')
    
    print(f"Gender unikal qiymatlari (keyin): {df[col].unique()}")
    return df

In [33]:
#2. Course normallashtirish

def normalize_course(df, col='course'):
    if col not in df.columns:
        print(f"'{col}' ustuni topilmadi.")
        return df
    
    print(f"Course unikal qiymatlari (oldin): {df[col].unique()}")
    
    # Kichik harf va tozalash
    df[col] = df[col].astype(str).str.lower().str.strip()
    
    # Mapping funksiyasi
    def map_course(val):
        if val in ['nan', 'none', '']:
            return 'Other'
        # Data Science ga tegishli kalit so'zlar
        ds_keywords = ['data science', 'data scientist', 'ml', 'machine learning', 
                       'ai', 'artificial intelligence', 'data']
        for kw in ds_keywords:
            if kw in val:
                return 'Data Science'
        # Python ga tegishli kalit so'zlar
        py_keywords = ['python', 'django', 'flask', 'py']
        for kw in py_keywords:
            if kw in val:
                return 'Python'
        # Boshqa hamma narsa 'Other'
        return 'Other'
    
    df[col] = df[col].apply(map_course)
    print(f"Course unikal qiymatlari (keyin): {df[col].unique()}")
    return df

In [34]:
#3. Status normallashtirish
def normalize_status(df, col='status'):
    if col not in df.columns:
        print(f"'{col}' ustuni topilmadi.")
        return df
    
    print(f"Status unikal qiymatlari (oldin): {df[col].unique()}")
    
    # Kichik harf va tozalash
    df[col] = df[col].astype(str).str.lower().str.strip()
    
    # Status mapping (agar aniq bir xil bo'lishi kerak bo'lsa)
    status_map = {
        'active': 'active', 'активный': 'active', 'act': 'active',
        'inactive': 'inactive', 'неактивный': 'inactive', 'inact': 'inactive',
        'completed': 'completed', 'завершен': 'completed', 'done': 'completed',
        'pending': 'pending', 'ожидание': 'pending', 'waiting': 'pending',
        'unknown': 'unknown', '': 'unknown', 'nan': 'unknown', 'none': 'unknown'
    }
    
    # Map orqali o'zgartirish, topilmaganlarni 'unknown' qilish
    df[col] = df[col].map(status_map).fillna('unknown')
    
    print(f"Status unikal qiymatlari (keyin): {df[col].unique()}")
    return df

In [35]:
df = normalize_gender(df, 'gender')
df = normalize_course(df, 'course')
df = normalize_status(df, 'status')

print(df[['gender', 'course', 'status']].head(10))

Gender unikal qiymatlari (oldin): ['fmale' 'Female' 'Male' 'FEMALE' 'femlae' 'MALE' 'male' 'female']
Gender unikal qiymatlari (keyin): ['Unknown' 'Female' 'Male']
Course unikal qiymatlari (oldin): ['Data Science' 'DATA SCIENCE' 'data-sciens' 'data_sciense' 'data science'
 'python' 'ds' 'd.s.' 'PYTHNO' 'Pyhton']
Course unikal qiymatlari (keyin): ['Data Science' 'Python' 'Other']
Status unikal qiymatlari (oldin): ['active' 'pending' 'inactive']
Status unikal qiymatlari (keyin): ['active' 'pending' 'inactive']
    gender        course    status
0  Unknown  Data Science    active
1   Female  Data Science    active
2  Unknown  Data Science   pending
3  Unknown  Data Science  inactive
4   Female  Data Science   pending
5     Male  Data Science   pending
6   Female  Data Science    active
7   Female        Python    active
8  Unknown         Other    active
9  Unknown  Data Science  inactive


In [36]:
#Step 9: Final type conversion va export

print("=== MA'LUMOT TURLARI (oldingi) ===")
print(df.dtypes)

=== MA'LUMOT TURLARI (oldingi) ===
student_id                       int64
name                            object
age                              Int64
gender                          object
score                          float64
phone                           object
city                            object
email                           object
date_of_join            datetime64[ns]
course                          object
attendance                       Int64
status                          object
gpa                            float64
remarks                         object
money_spent                    float64
event_time              datetime64[ns]
address_raw                     object
profile_json                    object
email_valid                       bool
email_status                    object
phone_digits                    object
phone_formatted                 object
phone_operator                  object
hobbies                         object
skills_tech_python           

In [37]:
# 1.1 Sana ustunlarini datetime ga o'tkazish (agar hali o'tkazilmagan bo'lsa)
date_columns = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        print(f"'{col}' datetime ga o'tkazildi.")

'date_of_join' datetime ga o'tkazildi.
'event_time' datetime ga o'tkazildi.


In [39]:
# 1.2 Raqamli ustunlarni tekshirish (agar kerak bo'lsa, qayta o'tkazish)
# Step 3 da bajarilgan bo'lishi kerak, lekin ishonch hosil qilish uchun:
numeric_cols = ['age', 'score', 'gpa', 'attendance', 'money_spent']
for col in numeric_cols:
    if col in df.columns and df[col].dtype not in ['int64', 'float64', 'Int64']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        if col in ['age', 'attendance']:
            df[col] = df[col].round().astype('Int64')
        else:
            df[col] = df[col].astype(float)
        print(f"'{col}' raqamli turga o'tkazildi.")

In [40]:
# 1.3 String ustunlar (object) – ularni object qoldiramiz, lekin bo'sh joylarni olib tashlash
string_cols = df.select_dtypes(include=['object']).columns
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()
    # (agar kerak bo'lsa, bo'sh qiymatlarni belgilash)
    df[col] = df[col].replace(['nan', 'none', ''], pd.NA)

print("\n=== MA'LUMOT TURLARI (keyingi) ===")
print(df.dtypes)


=== MA'LUMOT TURLARI (keyingi) ===
student_id                       int64
name                            object
age                              Int64
gender                          object
score                          float64
phone                           object
city                            object
email                           object
date_of_join            datetime64[ns]
course                          object
attendance                       Int64
status                          object
gpa                            float64
remarks                         object
money_spent                    float64
event_time              datetime64[ns]
address_raw                     object
profile_json                    object
email_valid                       bool
email_status                    object
phone_digits                    object
phone_formatted                 object
phone_operator                  object
hobbies                         object
skills_tech_python          

In [ ]:
#2. Sanalarni bir xil formatga keltirish (YYYY-MM-DD HH:MM:SS)

for col in date_columns:
    if col in df.columns:
        # Formatlangan versiyasini qo'shish (agar kerak bo'lsa)
        df[col + '_formatted'] = df[col].dt.strftime('%Y-%m-%d %H:%M:%S')
        

In [42]:
#3. CSV faylga saqlash

# Saqlashdan oldin barcha NaN qiymatlarni 'null' yoki bo'sh qoldirishni tanlash mumkin


output_filename = 'super_dirty_students_cleaned.csv'
df.to_csv(output_filename, index=False, encoding='utf-8')

print(f"\nTozalangan ma'lumotlar '{output_filename}' fayliga saqlandi.")
print(f"Jami qatorlar: {len(df)}, ustunlar: {len(df.columns)}")


Tozalangan ma'lumotlar 'super_dirty_students_cleaned.csv' fayliga saqlandi.
Jami qatorlar: 568, ustunlar: 45


In [49]:
#Step 10: QA checks

import pandas as pd

def qa_checks(original_df=None, cleaned_df=None, original_file=None, cleaned_file=None):
    """
    Sifat nazorati:
    - original va cleaned qatorlar soni
    - missing email va phone
    - numeric columnlar diapazoni (GPA, attendance, score)
    - dublikatlar
    """
    # Agar fayl nomlari berilgan bo'lsa, ularni yuklash
    if original_file:
        original_df = pd.read_csv(original_file, encoding='utf-8')
    if cleaned_file:
        cleaned_df = pd.read_csv(cleaned_file, encoding='utf-8')
    
    if cleaned_df is None:
        print("Tozalangan DataFrame kerak!")
        return
    
    print("="*60)
    print("QA TEKSHIRUV NATIJALARI")
    print("="*60)
    
    # 1. Qatorlar soni
    print(f"\n1. QATORLAR SONI:")
    if original_df is not None:
        print(f"   Original: {len(original_df)}")
    print(f"   Tozalangan: {len(cleaned_df)}")
    if original_df is not None:
        diff = len(original_df) - len(cleaned_df)
        print(f"   Farq: {diff} (agar musbat bo'lsa, dublikatlar o'chirilgan)")
    
    # 2. Missing email va phone
    print(f"\n2. BO'SH QIYMATLAR (TOZALANGAN):")
    if 'email' in cleaned_df.columns:
        missing_email = cleaned_df['email'].isna().sum()
        print(f"   Email: {missing_email} ({missing_email/len(cleaned_df)*100:.1f}%)")
    if 'phone' in cleaned_df.columns:
        missing_phone = cleaned_df['phone'].isna().sum()
        print(f"   Phone: {missing_phone} ({missing_phone/len(cleaned_df)*100:.1f}%)")
    
    # 3. Numeric column diapazonlari
    print(f"\n3. NUMERIC COLUMN DIAPAZONLARI:")
    numeric_checks = {
        'gpa': (0.0, 4.0),
        'attendance': (0, 100),
        'score': (0, 100)
    }
    for col, (low, high) in numeric_checks.items():
        if col in cleaned_df.columns:
            col_min = cleaned_df[col].min()
            col_max = cleaned_df[col].max()
            col_mean = cleaned_df[col].mean()
            print(f"   {col}: min={col_min:.2f}, max={col_max:.2f}, mean={col_mean:.2f}")
            # Diapazondan tashqari qiymatlar
            outliers = cleaned_df[(cleaned_df[col] < low) | (cleaned_df[col] > high)][col].count()
            if outliers > 0:
                print(f"     ⚠️  {outliers} ta qiymat diapazondan tashqari ({low}-{high})")
            else:
                print(f"     ✅ Barcha qiymatlar {low}-{high} oralig'ida")
    
    # 4. Dublikatlar
    print(f"\n4. DUBLIKATLAR:")
    dup_count = cleaned_df.duplicated().sum()
    if dup_count == 0:
        print(f"   ✅ To'liq dublikatlar yo'q")
    else:
        print(f"   ⚠️  {dup_count} ta to'liq dublikat topildi!")
    
    if 'email' in cleaned_df.columns:
        email_dup = cleaned_df.duplicated(subset=['email']).sum()
        if email_dup == 0:
            print(f"   ✅ Email dublikatlari yo'q")
        else:
            print(f"   ⚠️  {email_dup} ta email dublikati bor")
    
    # Qo'shimcha: age tekshiruvi (agar mavjud bo'lsa)
    if 'Age' in cleaned_df.columns:
        invalid_age = cleaned_df[(cleaned_df['age'] < 0) | (cleaned_df['age'] > 120)]['age'].count()
        if invalid_age > 0:
            print(f"   ⚠️  age: {invalid_age} ta noto'g'ri qiymat (0-120 oralig'ida emas)")
    
    print("\n" + "="*60)
    print("QA TEKSHIRUV TUGADI")
    print("="*60)
    

In [50]:
qa_checks(original_file='super_dirty_students.csv', 
          cleaned_file='super_dirty_students_cleaned.csv')

QA TEKSHIRUV NATIJALARI

1. QATORLAR SONI:
   Original: 1000
   Tozalangan: 568
   Farq: 432 (agar musbat bo'lsa, dublikatlar o'chirilgan)

2. BO'SH QIYMATLAR (TOZALANGAN):
   Email: 0 (0.0%)
   Phone: 0 (0.0%)

3. NUMERIC COLUMN DIAPAZONLARI:
   gpa: min=-2.00, max=37.00, mean=12.00
     ⚠️  433 ta qiymat diapazondan tashqari (0.0-4.0)
   attendance: min=60.00, max=120.00, mean=103.20
     ⚠️  434 ta qiymat diapazondan tashqari (0-100)
   score: min=50.00, max=100.00, mean=75.49
     ✅ Barcha qiymatlar 0-100 oralig'ida

4. DUBLIKATLAR:
   ✅ To'liq dublikatlar yo'q
   ✅ Email dublikatlari yo'q

QA TEKSHIRUV TUGADI
